In [1]:
import tempfile
import os
from ansys.aedt.core import Desktop, generate_unique_name
import optuna
from optuna.samplers import TPESampler
import sys
import numpy as np
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..'))) 
import script_otim as script
import logging

c:\Users\luis.costa\.conda\envs\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
optuna.logging.get_logger("optuna").addHandler(logging.StreamHandler(sys.stdout))
study_name = "tpe_seed1" 
storage_name = "sqlite:///{}.db".format(study_name)

In [3]:
tmpfold = tempfile.gettempdir()

temp_folder = os.path.join(
    tmpfold,
    generate_unique_name("Antenna")
)

if not os.path.exists(temp_folder):
    os.mkdir(temp_folder)

non_graphical = os.getenv(
    "PYAEDT_NON_GRAPHICAL",
    "True"
).lower() in ("true", "1", "t")

d = Desktop(
    "2025.1",
    non_graphical=non_graphical,
    new_desktop=True
)


PyAEDT INFO: Python version 3.10.18 | packaged by conda-forge | (main, Jun  4 2025, 14:42:04) [MSC v.1943 64 bit (AMD64)].
PyAEDT INFO: PyAEDT version 0.20.1.
PyAEDT INFO: Initializing new Desktop session.
PyAEDT INFO: Log on console is enabled.
PyAEDT INFO: Log on file C:\Users\LUIS~1.COS\AppData\Local\Temp\pyaedt_luis.costa_6b9e1f32-16ac-4113-ade4-4c6e38eb1d47.log is enabled.
PyAEDT INFO: Log on AEDT is disabled.
PyAEDT INFO: Debug logger is disabled. PyAEDT methods will not be logged.
PyAEDT INFO: Launching PyAEDT with gRPC plugin.
PyAEDT INFO: New AEDT session is starting on gRPC port 53818.
PyAEDT INFO: Electronics Desktop started on gRPC port: 53818 after 11.797143697738647 seconds.
PyAEDT INFO: AEDT installation Path C:\Program Files\ANSYS Inc\v251\AnsysEM
PyAEDT INFO: Ansoft.ElectronicsDesktop.2025.1 version started with process ID 18852.


In [4]:
def objective(trial):
    a1 = trial.suggest_float("a1", 1.0, 1.0)
    a2 = trial.suggest_float("a2", 1.0, 1.0)
    n1 = trial.suggest_int("n1", 1, 100)
    n2 = trial.suggest_float("n2", 1.0, 10.0)
    n3 = trial.suggest_float("n3", 1.0, 10.0)
    n4 = trial.suggest_int("n4", 1, 100)
    perc = trial.suggest_float("perc", 0.2, 0.8)
    r = trial.suggest_int("r", 90, 150)

    df = script.makeAntenna(a1, a2, n1, n2, n3, n4, perc, r, temp_folder)
    df_filtrado = df[(df['Freq [GHz]'] >= 0.3) & (df['Freq [GHz]'] <= 1.5)]
    s11 = np.asarray(df_filtrado['|$S_{11}$| [dB]'], dtype=float)
    print(s11)

    proportion_bad = float(np.mean(s11 >= -10))

    return proportion_bad

study = optuna.create_study(
    directions=["minimize"],    
    study_name=study_name, 
    storage=storage_name,
    sampler=TPESampler(seed=123),
    load_if_exists=True
)

    

[I 2025-12-08 11:56:03,447] Using an existing study with name 'tpe_seed1' instead of creating a new one.


Using an existing study with name 'tpe_seed1' instead of creating a new one.


In [5]:
study.optimize(objective, n_trials=1000)

PyAEDT INFO: Python version 3.10.18 | packaged by conda-forge | (main, Jun  4 2025, 14:42:04) [MSC v.1943 64 bit (AMD64)].
PyAEDT INFO: PyAEDT version 0.20.1.
PyAEDT INFO: Returning found Desktop session with PID 18852!
PyAEDT INFO: Project Project1 has been created.
PyAEDT INFO: No design is present. Inserting a new design.
PyAEDT INFO: Added design 'HFSS_DXF' of type HFSS.
PyAEDT INFO: Aedt Objects correctly read
PyAEDT INFO: Modeler class has been initialized! Elapsed time: 0m 2sec
PyAEDT INFO: Materials class has been initialized! Elapsed time: 0m 0sec
PyAEDT INFO: Parsing design objects. This operation can take time
PyAEDT INFO: Refreshing bodies from Object Info
PyAEDT INFO: Bodies Info Refreshed Elapsed time: 0m 0sec
PyAEDT INFO: 3D Modeler objects parsed. Elapsed time: 0m 0sec
PyAEDT INFO: Deleted 1 Objects: cut.
PyAEDT INFO: Parsing design objects. This operation can take time
PyAEDT INFO: Refreshing bodies from Object Info
PyAEDT INFO: Bodies Info Refreshed Elapsed time: 0m 0

[W 2025-12-08 12:21:48,631] Trial 0 failed with parameters: {'a1': 1.0, 'a2': 1.0, 'n1': 70, 'n2': 3.575254014553415, 'n3': 3.041663082077828, 'n4': 56, 'perc': 0.6316813818713379, 'r': 115} because of the following error: KeyError('Freq [GHz]').
Traceback (most recent call last):
  File "c:\Users\luis.costa\.conda\envs\venv\lib\site-packages\pandas\core\indexes\base.py", line 3812, in get_loc
    return self._engine.get_loc(casted_key)
  File "pandas/_libs/index.pyx", line 167, in pandas._libs.index.IndexEngine.get_loc
  File "pandas/_libs/index.pyx", line 196, in pandas._libs.index.IndexEngine.get_loc
  File "pandas/_libs/hashtable_class_helper.pxi", line 7088, in pandas._libs.hashtable.PyObjectHashTable.get_item
  File "pandas/_libs/hashtable_class_helper.pxi", line 7096, in pandas._libs.hashtable.PyObjectHashTable.get_item
KeyError: 'Freq [GHz]'

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "c:\Users\luis.costa\.con

Trial 0 failed with parameters: {'a1': 1.0, 'a2': 1.0, 'n1': 70, 'n2': 3.575254014553415, 'n3': 3.041663082077828, 'n4': 56, 'perc': 0.6316813818713379, 'r': 115} because of the following error: KeyError('Freq [GHz]').
Traceback (most recent call last):
  File "c:\Users\luis.costa\.conda\envs\venv\lib\site-packages\pandas\core\indexes\base.py", line 3812, in get_loc
    return self._engine.get_loc(casted_key)
  File "pandas/_libs/index.pyx", line 167, in pandas._libs.index.IndexEngine.get_loc
  File "pandas/_libs/index.pyx", line 196, in pandas._libs.index.IndexEngine.get_loc
  File "pandas/_libs/hashtable_class_helper.pxi", line 7088, in pandas._libs.hashtable.PyObjectHashTable.get_item
  File "pandas/_libs/hashtable_class_helper.pxi", line 7096, in pandas._libs.hashtable.PyObjectHashTable.get_item
KeyError: 'Freq [GHz]'

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "c:\Users\luis.costa\.conda\envs\venv\lib\site-packag

[W 2025-12-08 12:21:48,659] Trial 0 failed with value None.


Trial 0 failed with value None.


KeyError: 'Freq [GHz]'

In [ ]:
len(study.trials)

In [ ]:
study.best_params

In [ ]:
study.best_trial

In [ ]:
d.release_desktop()